# Workflow Versioning, Activation & Cloning

This notebook focuses on the **version lifecycle** of a workflow:

1. Every workflow starts with an automatic version `0` (the working copy)
2. Snapshotting the current design as an immutable version
3. Editing the working copy and diffing it against a snapshot
4. Activating (promoting / rolling back to) a version
5. Renaming, cloning, and deleting versions

> **Mental model:** version `0` is the *working copy* you edit. `versions.create()` freezes a snapshot of the current design into a new, higher-numbered version. Later edits only touch the working copy — that is what lets you diff against, and roll back to, a snapshot.

> **Tip:** configs are built with the typed classes from `interactly.configs` (requires `pip install "interactly[configs]"`).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Create a workflow

A brand-new workflow automatically gets version `0` ("Initial Version"), which is immediately the **active** version.

In [ ]:
from interactly.configs import (
    DirectEdgeConfig,
    SayLLMNodeConfig,
    PromptConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

greeting_node = SayLLMNodeConfig(
    name="Greet User",
    is_start=True,
    main_response_config=PromptConfig(
        prompt="You are a friendly support agent. Greet the user warmly and ask how you can help.",
    ),
)

farewell_node = SayLLMNodeConfig(
    name="Farewell User",
    main_response_config=PromptConfig(
        prompt="You are a friendly support agent. Say goodbye warmly and thank the user for their time.",
    ),
    self_loop=False,
    wait_for_user_message=False,
)

config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Customer Support Agent",
        description="Two-node support workflow used to demo versioning.",
    ),
    node_configs=[greeting_node, farewell_node],
    edge_configs=[
        DirectEdgeConfig(
            source_node_logical_id=greeting_node.logical_id,
            destination_node_logical_id=farewell_node.logical_id,
            name="Greeting -> Farewell",
        )
    ],
)

workflow: Workflow = await client.workflows.create_from_config(config)

WF_ID = workflow.id
print(f"Created workflow  id={WF_ID}  name={workflow.name!r}")
print(f"Active version number: {workflow.active_version_number}")

## 2. Inspect the initial version

`versions.list()` returns every version for the workflow. The **active** version is identified by the workflow's `active_version_number` (the source of truth), so we use a small helper to flag it.

In [ ]:
from typing import List, Optional
from interactly.types.workflows.workflow import WorkflowVersion


async def show_versions(active_number: int, workflow_id: Optional[str] = None) -> None:
    """Print all versions of a workflow, flagging the active one.

    Defaults to the workflow created above (``WF_ID``); pass ``workflow_id`` to
    inspect a different workflow, e.g. a clone.
    """
    versions: List[WorkflowVersion] = await client.workflows.versions.list(workflow_id or WF_ID)
    for v in sorted(versions, key=lambda x: x.version_number):
        marker = "  <-- ACTIVE" if v.version_number == active_number else ""
        print(f"  v{v.version_number}  {v.version_name!r}{marker}")


await show_versions(workflow.active_version_number)

## 3. Snapshot the current design as a version

`versions.create()` freezes the current working copy into a new, higher-numbered version. We keep `mark_as_active=False` so version `0` stays active while we keep editing the working copy.

In [ ]:
v1: WorkflowVersion = await client.workflows.versions.create(
    WF_ID,
    version_name="v1 - snapshot of original",
)
V1_NUMBER = v1.version_number
print(f"Snapshotted version {V1_NUMBER}  name={v1.version_name!r}")

# Version 0 is still the active working copy.
await show_versions(workflow.active_version_number)

## 4. Edit the working copy

Now we change the greeting node's prompt. Edits apply to the **working copy** (version `0`); the `v1` snapshot stays frozen.

> **Note:** `create_from_config()` assigns fresh server-side `logical_id`s to nodes, so we locate the node by its **name** rather than the client-side `logical_id`.

In [ ]:
# Locate the greeting node by name (logical_ids are remapped server-side).
nodes_page = await client.nodes.list(workflow_id=WF_ID)
nodes = await nodes_page.list_all()
greeting_db_node = next(n for n in nodes if n.node_config.name == "Greet User")

await client.nodes.update(
    node_id=greeting_db_node.id,
    node_config=SayLLMNodeConfig(
        main_response_config=PromptConfig(
            prompt="You are an expert support agent. Greet warmly and collect the issue details up front.",
        ),
    ),
)
print("Updated the greeting prompt on the working copy (v0).")

## 5. Diff the snapshot against the working copy

`versions.diff()` compares two versions and returns a typed `VersionDiff`. Comparing the frozen `v1` snapshot with the edited working copy (`0`) surfaces the change.

The edited greeting node is reported with `status="modified"`, and its `changes` list carries the exact field-level diff — here a `main_response_config.prompt` change with its old and new values. (Snapshots preserve each node's `logical_id`, so an in-place edit is matched as *modified* rather than as an add/remove pair.)

In [ ]:
import json

from interactly import VersionDiff

# `diff()` returns a typed VersionDiff. `summary` has the aggregate counts;
# `nodes`/`edges` are per-entity diffs, and each modified entity's `changes`
# lists the exact field-level edits (path + old value -> new value).
diff: VersionDiff = await client.workflows.versions.diff(WF_ID, V1_NUMBER, 0)
print("Summary:", json.dumps(diff.summary, indent=2))

print("\nNode changes:")
for node in diff.nodes:
    print(f"  [{node.status}] {node.name!r} ({node.node_type})")
    for change in node.changes:
        print(f"      {change.path}: {change.old_value!r} -> {change.new_value!r}")

if diff.edges:
    print("\nEdge changes:")
    for edge in diff.edges:
        print(f"  [{edge.status}] {edge.name!r}")
        for change in edge.changes:
            print(f"      {change.path}: {change.old_value!r} -> {change.new_value!r}")

## 6. Activate a version

`versions.activate()` promotes a version to be the one served at runtime. Here we activate the frozen `v1` snapshot — effectively rolling back the live workflow. The returned `Workflow` reflects the new `active_version_number`.

In [ ]:
activated: Workflow = await client.workflows.versions.activate(WF_ID, V1_NUMBER)
print(f"Active version number is now: {activated.active_version_number}")

await show_versions(activated.active_version_number)

## 7. Rename a version

`versions.update()` renames a version without touching its snapshotted design.

In [ ]:
renamed: WorkflowVersion = await client.workflows.versions.update(
    WF_ID,
    0,
    version_name="v0 - working copy",
)
print(f"Renamed version 0 -> {renamed.version_name!r}")

await show_versions(activated.active_version_number)

## 8. Clone the workflow

`workflows.clone()` deep-copies a workflow (nodes, edges, and versions) into a brand-new workflow with its own id.

In [ ]:
clone: Workflow = await client.workflows.clone(WF_ID, name="Customer Support Agent - Copy")
CLONE_ID = clone.id
print(f"Cloned workflow  id={CLONE_ID}  name={clone.name!r}")

In [ ]:
# The clone is an independent workflow with its own id. `clone()` copies every version
# by default (clone_all_versions=True), so pass clone.id to list the CLONE's versions
# (not WF_ID, the original).
print(f"Cloned workflow {CLONE_ID} versions:")
await show_versions(clone.active_version_number, clone.id)

## 9. Clone select versions of a workflow

`workflows.clone()` copies **every** version by default. To copy only a subset, pass
`version_numbers=[...]` — the server requires at least one version to be selected. Here we
clone just version `0`.

In [ ]:
# Clone only version 0. Passing `version_numbers` selects a subset (and sets
# clone_all_versions=False under the hood); the server requires at least one version.
# The source's active version (1) isn't in the selection, so the clone falls back to
# activating the lowest cloned version (0).
subset_clone: Workflow = await client.workflows.clone(
    WF_ID,
    name="Customer Support Agent - v0 only",
    version_numbers=[0],
)
SUBSET_CLONE_ID = subset_clone.id
print(f"Cloned workflow  id={SUBSET_CLONE_ID}  name={subset_clone.name!r}")
print(f"Active version number: {subset_clone.active_version_number}")

In [ ]:
# The subset clone contains only the version(s) we selected.
print(f"Subset-cloned workflow {SUBSET_CLONE_ID} versions:")
await show_versions(subset_clone.active_version_number, subset_clone.id)

## 10. Delete a version

`versions.delete()` removes a specific version. You can delete any non-active version; here we delete the working copy `0` now that `v1` is active.

In [ ]:
await client.workflows.versions.delete(WF_ID, 0)
print("Deleted version 0.")

await show_versions(activated.active_version_number)

## 11. Cleanup

In [ ]:
await client.workflows.delete(CLONE_ID)
await client.workflows.delete(SUBSET_CLONE_ID)
await client.workflows.delete(WF_ID)
print("All workflows deleted.")